# AI Ignite at York - Forge Track
## Emotion Classification Workshop - Base Notebook

**Workshop:** BiLSTM vs BERT vs LLMs for Emotion Classification  
**Datasets:** EmoCAP, EMLD  
**Leads:** Sagar, Mehwish, Davit, Ali, Sheheryar  

> **How to use this notebook:**  
> This is the shared base everyone starts from. It sets up your environment, folder structure, and loads the datasets.  
> Each lead has their own notebook (in their own folder) that imports from here.

---

## Step 1 - Folder Structure Setup
Run this once to create the required folders on your machine.

In [ ]:
import os

folders = [
    "data/raw",
    "data/processed",
    "notebooks/davit",
    "notebooks/ali",
    "notebooks/sheheryar",
    "outputs/predictions",
    "outputs/reports",
    "outputs/figures",
    "slides",
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("Folder structure created:")
for folder in folders:
    print(f"  {folder}")

## Step 2 - Install Dependencies
Run this once at the start of the workshop (or in advance if possible).

In [ ]:
# Run this cell to install all required packages
import subprocess, sys

packages = [
    "pandas",
    "numpy",
    "matplotlib",
    "seaborn",
    "scikit-learn",
    "transformers",
    "torch",
    "datasets",
    "openai",          # for zero-shot / few-shot (Davit)
    "imbalanced-learn",# for data balancing (Davit)
    "tqdm",
]

for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])
    print(f"{pkg} installed")

## Step 3 - Imports
All shared imports used across the workshop.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split

# Suppress warnings for cleaner output
import warnings
warnings.filterwarnings('ignore')

print("All imports loaded successfully")

## Step 4 - Load Datasets

> **Mehwish** will prepare the cleaned dataset files and upload them to the `/data/processed/` folder in GitHub.  
> Once she is done, this cell will just work for everyone.

**EmoCAP** - conversation-level emotion dataset  
**EMLD** - multi-label emotion dataset  

Each dataset has these columns after preprocessing:
- `utterance` - the text
- `label` - the emotion (e.g. happy, sad, angry, neutral, fear, surprise, disgust)

In [ ]:
# Load EmoCAP
emocap = pd.read_csv("data/processed/emocap_clean.csv")
print(f"EmoCAP loaded: {len(emocap)} rows")
print(emocap.head())
print("\nEmotion labels:", emocap['label'].unique())

In [ ]:
# Load EMLD
emld = pd.read_csv("data/processed/emld_clean.csv")
print(f"EMLD loaded: {len(emld)} rows")
print(emld.head())
print("\nEmotion labels:", emld['label'].unique())

## Step 5 - Class Distribution Check
Always check the class balance before training anything. Imbalanced data = biased models.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, df, name in zip(axes, [emocap, emld], ["EmoCAP", "EMLD"]):
    counts = df['label'].value_counts()
    ax.bar(counts.index, counts.values, color='#534AB7', edgecolor='white')
    ax.set_title(f"{name} — Class Distribution", fontsize=13, fontweight='bold')
    ax.set_xlabel("Emotion")
    ax.set_ylabel("Count")
    ax.tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig("outputs/figures/class_distribution.png", dpi=150)
plt.show()
print("Saved to outputs/figures/class_distribution.png")

## Step 6 - Train / Test Split
Split the data before any modelling. Use the same split across all approaches so results are comparable.

In [ ]:
# Using EmoCAP as the primary dataset
X = emocap['utterance'].values
y = emocap['label'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training samples : {len(X_train)}")
print(f"Test samples     : {len(X_test)}")
print("\nUse X_train, X_test, y_train, y_test in your individual notebooks")

## Step 7 - Output Format (IMPORTANT {everyone must follow this)

After running your model, save predictions in this exact format so Sheheryar's evaluation notebook can load them.

**Required columns:** `utterance`, `true_label`, `predicted_label`, `model_name`

In [ ]:
# TEMPLATE - replace y_pred and model_name with your actual values

def save_predictions(X_test, y_test, y_pred, model_name: str):
    """
    Save model predictions in the standard format.
    Call this at the end of your notebook.
    
    Args:
        X_test      : list of utterances
        y_test      : list of true labels
        y_pred      : list of predicted labels
        model_name  : e.g. 'zero_shot', 'few_shot', 'bilstm', 'bert'
    """
    df = pd.DataFrame({
        "utterance"       : X_test,
        "true_label"      : z_test,
        "predicted_label" : z_pred,
        "model_name"      : model_name,
    })
    path = f"outputs/predictions/{model_name}_predictions.csv"
    df.to_csv(path, index=False)
    print(f"Predictions saved to {path}")
    return df

# Example usage (replace with your real predictions):
# save_predictions(X_test, y_test, y_pred, model_name="bert")

---
## Folder Reference

> Each folder has a specific purpose. Do not save files outside their intended location.

| Folder | Contents |
|---|---|
| `data/raw/` | Original dataset files (do not modify) |
| `data/processed/` | Cleaned CSVs prepared by Mehwish |
| `notebooks/davit/` | Zero-shot, few-shot and data balancing |
| `notebooks/ali/` | Fine-tuning - BiLSTM and BERT |
| `notebooks/sheheryar/` | Evaluation pipeline and results comparison |
| `outputs/predictions/` | Predictions CSVs (one per model) |
| `outputs/reports/` | Classification reports |
| `outputs/figures/` | Charts and confusion matrices |
| `slides/` | Presentation decks |

---

**AI Ignite at York - Forge Track - October 2026**  
Questions? Contact Sagar: sagarh@my.yorku.ca